<div dir="rtl" lang="he" style="text-align: right;">

# מפגש 11 — Boxplot ואיכות נתונים
2 שעות הוראה ותרגול ו־1 שעות פרויקט; יחד 135 דקות.

## סיפור המסגרת
זמן משלוח חריג יכול להיות פקק אמיתי או טעות הקלדה. שתי הזמנות דומות יכולות להיות עסקאות נפרדות. נבדוק לפני ניקוי.

## מטרות
- לקרוא רבעונים ושפמים.
- לבדוק זהות וסבירות.
- לתעד ניקוי.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## הנחיות למורה
זהו פתרון מתועד למעבדה. מבדילים בין תוצאה נכונה לבין נימוק נכון, ובודקים שהקוד תואם לנתוני התלמיד. המקורות הישנים נשמרו ללא שינוי.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## פתרון חלק 1
הפתרון הבא תואם לנתוני התרגול העצמאי, ולא לנתוני הדוגמה הפתורה.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

### 1–2. דוגמה
חציון 14; IQR=16−12=4. גבולות 6 ו־ 22. קצות שפמים 10 ו־ 16, שהם ערכים נצפים בתוך הגבולות. 30 מעבר לגבול, אך אין להסיק מכך טעות בנתון המומצא.

</div>

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
demo_df = pd.DataFrame({"value": [10, 12, 14, 16, 30]})
q1 = demo_df["value"].quantile(0.25, interpolation="linear")
q2 = demo_df["value"].median()
q3 = demo_df["value"].quantile(0.75, interpolation="linear")
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
print("Q1:", q1, "Q2:", q2, "Q3:", q3, "IQR:", iqr)
print("גבולות:", lower_fence, upper_fence)
plt.figure(figsize=(7, 4))
sns.boxplot(data=demo_df, y="value", whis=1.5, color="lightblue")
plt.title("Teaching example: fences are not whisker endpoints")
plt.xlabel("All teaching observations")
plt.ylabel("Value")
plt.tight_layout()
plt.show()

In [ ]:
url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
penguins_df = pd.read_csv(url)
species_order = ["Adelie", "Chinstrap", "Gentoo"]
plt.figure(figsize=(7, 4))
sns.boxplot(data=penguins_df, x="species", y="flipper_length_mm", order=species_order, whis=1.5, color="lightblue")
plt.title("Flipper length distributions by species")
plt.xlabel("Species")
plt.ylabel("Flipper length (mm)")
plt.tight_layout()
plt.show()

<div dir="rtl" lang="he" style="text-align: right;">

### 4–5. כרטיס לדוגמה
שאלה: כיצד מרכז ופיזור אורך הסנפיר שונים בין מינים בקובץ? המשתנה כמותי רציף במ״מ; קבוצות species. חציון Gentoo גבוה יותר בתמונה, ויש להשוות גם פיזור וחפיפה של הערכים הידועים. זה אינו אומר שכל Gentoo ארוך־סנפיר מכל פרט במינים האחרים.

גובה הקופסה מציג IQR, לא ספירה; הרבעונים מחושבים בתוך כל מין. נקודה שמסומנת לפי 1.5×IQR דורשת בירור יחידה, מקור ומצב התצפית, ולא מחיקה אוטומטית. לא בוצע מבחן מובהקות או נורמליות. חסרים אינם מוצגים והמקור נשאר.

</div>

In [ ]:
print("ממדי המקור:", penguins_df.shape)

<div dir="rtl" lang="he" style="text-align: right;">

### רשות — קבוצה
מחשבים על המדידות הידועות של Adelie. בשיטות חישוב רבעונים שונות עשויים להיות הבדלים; אנו מציינים linear לתאימות לחישוב המוצג.

</div>

In [ ]:
adelie_mask = penguins_df["species"] == "Adelie"
adelie_df = penguins_df[adelie_mask]
adelie_q1 = adelie_df["flipper_length_mm"].quantile(0.25, interpolation="linear")
adelie_q2 = adelie_df["flipper_length_mm"].median()
adelie_q3 = adelie_df["flipper_length_mm"].quantile(0.75, interpolation="linear")
print("רבעוני Adelie בממ:", adelie_q1, adelie_q2, adelie_q3)
print("IQR בממ:", adelie_q3 - adelie_q1)

<div dir="rtl" lang="he" style="text-align: right;">

## פתרון חלק 2
הפתרון הבא תואם לנתוני התרגול העצמאי, ולא לנתוני הדוגמה הפתורה.

</div>

In [ ]:
import pandas as pd
import numpy as np
raw = pd.DataFrame({
 "order_id":[201,202,203,203,204,205,206],
 "ordered_at":["2026-02-01","2026-02-02","2026-02-03","2026-02-03","2026-02-04","2026-02-05","2026-02-06"],
 "items":[1,2,3,3,2,1,2],"unit_price":[20.,10.,5.,5.,10.,20.,10.],
 "duration_min":[5.,9.,12.,12.,0.,-1.,None]
})

<div dir="rtl" lang="he" style="text-align: right;">

## ניקוי לפי חוזה
בחוזה התרגול 0 אינו תקין, בניגוד למקרים שבהם משך אפס אפשרי. לא מוחקים הזמנות שונות בגלל שוויון מדידות.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## פירוש ומגבלה
לא מחליפים את כל האפסים ב־ NaN; לא מוחקים כל חריג; כשאותו מזהה מופיע עם מידע סותר צריך בירור ולא keep-first שקט. כאן שורת הייבוא זהה ולכן הכלל מוצדק. למתקשים עובדים עם שלוש שורות בלבד, למצטיינים מוסיפים מקרה סתירה ומנסחים מדיניות בלי למחוק.
המספרים נקראים מפלט ההרצה; אין לדרוש ציון ביצועים מסוים. הסבר חלופי מנומק יכול להיות נכון.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## תשובה לכרטיס היציאה
לא. בודקים מזהה ומשמעות. שתי עסקאות שונות עשויות להיות זהות בשדות האלה.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## משוב דידקטי
בדקו בנפרד: בחירת פעולה, תקינות קוד, פירוש פלט, יחידות ומגבלה. שאלו מה ישתנה בקלט חדש. אם יש קושי, חזרו לדוגמה קטנה לפני הגדלת המאגר. זהו משוב מקומי, לא מחוון ניקוד רשמי.

</div>